# 23B Harness: identity, context and the channel
Who may write each file, what enters the model's context, and who may give instructions.

In [ ]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

In [ ]:
print(pd.DataFrame([(f, a, w) for f, (a, w) in workspace.AUTHORITY.items()], columns=["file", "authority", "written by"]).to_string(index=False))

In [ ]:
WS = workspace.fresh_workspace(ROOT / P["agent"]["seed"], ROOT / ".cache" / "nb23")   # a scratch copy: reruns start clean
idx = skills.skill_index(skills.load_skills(WS / "skills"))
ctx, report = workspace.assemble(workspace.load_parts(WS, "mandate_a", skills_index=idx), P["context"]["budgets"])
print(pd.DataFrame(report).T)
print("\nTokens in the assembled context:", sum(r["used"] for r in report.values()))

In [ ]:
CAN = json.load(open(FX / "canaries.json"))
print("Mandate A canaries in A's context:", [c for _, c in evals.canary_hits([ctx], CAN["mandate_a"])])
print("Mandate B canaries in A's context:", evals.canary_hits([ctx], CAN["mandate_b"]), "(must be empty)")
print("Mentions Northgate:", "Northgate" in ctx)

In [ ]:
inbox = [json.loads(l) for l in (FX / "inbox.jsonl").read_text().splitlines()]
for m in inbox:
    print(f"{m['message_id']} from {m['from']:32s} -> {channel.authority(m, USERS)}")
c = json.loads((FX / "corrections.jsonl").read_text().splitlines()[0])
print(f"{c['message_id']} from Priya (session token) -> {channel.authority(c, USERS)}")
print("\nm-001 asks:", inbox[0]["body"])

The client's email asks what another client holds. It is **data**: the agent can report that the client asked, and must never answer it. Only Priya, authenticated by her session token, gives instructions.

**Your turn.** Build the CLI and web chat (Part 0): the session token authenticates Priya, the session is bound to one mandate, and a test shows mandate A's assembled context contains no mandate B content or canaries.